##introduction to data ingestion

In [4]:
import os
from typing import List, Dict, Any
import pandas

In [8]:
from langchain_core.documents import Document

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print("Setup Completed")

d:\RAG UDEMY\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup Completed


## Understanding Document Structure in Langchain

In [12]:
##Create a simple document
doc=Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "Source":"emanple.txt",
        "page":1,
        "author":"Sumona Nayek",
        "date_created":"2026-08-08",
        "custom_field":"any_value"
                  
     }
)
print("Documnet Structure")
print(f"Content :{doc.page_content}")
print(f"Metadata :{doc.metadata}")

Documnet Structure
Content :This is the main text content that will be embedded and searched.
Metadata :{'Source': 'emanple.txt', 'page': 1, 'author': 'Sumona Nayek', 'date_created': '2026-08-08', 'custom_field': 'any_value'}


In [13]:
type(doc)

langchain_core.documents.base.Document

## Text Files(.txt)-The Simple Case{#2-text-fiels}

In [15]:
## Create a simple txt file
import os
os.makedirs("data/text_fiels",exist_ok=True)

In [18]:
simple_texts={
    "data/text_fiels/python_intro.txt":"""Python is a high-level programming language
    

Python is widely used for:
- Data Science
- Machine Learning
- AI
- RAG applications

Variables:
name = "Sumona"
age = 25

Lists:
skills = ["Python", "SQL", "RAG"]
"""
}

for filepath,content in simple_texts.items():
    with open(filepath,'w',encoding='utf-8') as f:
        f.write(content)

print("Sample text file created!")        

Sample text file created!


## TextLoader- Read Single File

In [23]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader("data/text_fiels/python_intro.txt",encoding="utf-8")

documents=loader.load()
print(f"Length of documents: {len(documents)}")
print(f"First 100 characters: {documents[0].page_content[:100]}")
print(f"Metadata: {documents[0].metadata}")

Length of documents: 1
First 100 characters: Python is a high-level programming language


Python is widely used for:
- Data Science
- Machine Le
Metadata: {'source': 'data/text_fiels/python_intro.txt'}


## DirectoryLoader- Multiple Text Files

In [27]:
from langchain_community.document_loaders import DirectoryLoader
## load all the text files from the directory
dir_loader=DirectoryLoader(
    "data/text_fiels",
    glob="**/*.txt",##Patern to match files
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True

)
documents=dir_loader.load()
print(f"Document Loaded{len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument{i+1}:")
    print(f" Source: {doc.metadata['source']}")
    print(f" Length: {len(doc.page_content)} characters")

100%|██████████| 1/1 [00:00<00:00, 1206.65it/s]

Document Loaded1 documents

Document1:
 Source: data\text_fiels\python_intro.txt
 Length: 210 characters


## Text Splitter Statergies

In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print(documents)

[Document(metadata={'source': 'data\\text_fiels\\python_intro.txt'}, page_content='Python is a high-level programming language\n\n\nPython is widely used for:\n- Data Science\n- Machine Learning\n- AI\n- RAG applications\n\nVariables:\nname = "Sumona"\nage = 25\n\nLists:\nskills = ["Python", "SQL", "RAG"]\n')]


In [32]:
## Method 1- Character Text Splitter
text=documents[0].page_content
text

'Python is a high-level programming language\n\n\nPython is widely used for:\n- Data Science\n- Machine Learning\n- AI\n- RAG applications\n\nVariables:\nname = "Sumona"\nage = 25\n\nLists:\nskills = ["Python", "SQL", "RAG"]\n'

In [36]:
## Method 1- Character Text Splitter
from langchain_text_splitters import CharacterTextSplitter # CharacterTextSplitter ko import kar rahe hain

text_splitter = CharacterTextSplitter(
    separator="\n", # Text ko newline (\n) ke basis par split karega

    chunk_size=200, # Har chunk ki maximum target length 200 characters hogi

    chunk_overlap=20,# Consecutive chunks ke beech 20 characters overlap rahenge

    length_function=len # Chunk ki length calculate karne ke liye Python ka len() function use hoga
)

chunks=text_splitter.split_text(text)
print(f"Length of chunks: {len(chunks)}")

print(f"First chunk: {chunks[0][:100]}")

Length of chunks: 2
First chunk: Python is a high-level programming language
Python is widely used for:
- Data Science
- Machine Lear


In [39]:
print(chunks[0])
print("----------------")
print(chunks[1])

Python is a high-level programming language
Python is widely used for:
- Data Science
- Machine Learning
- AI
- RAG applications
Variables:
name = "Sumona"
age = 25
Lists:
----------------
age = 25
Lists:
skills = ["Python", "SQL", "RAG"]


In [41]:
## Method 1- Character Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter # CharacterTextSplitter ko import kar rahe hain

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," ",""], # Text ko newline (\n) ke basis par split karega

    chunk_size=200, # Har chunk ki maximum target length 200 characters hogi

    chunk_overlap=20,# Consecutive chunks ke beech 20 characters overlap rahenge

    length_function=len # Chunk ki length calculate karne ke liye Python ka len() function use hoga
)

recursive_chunks=recursive_splitter.split_text(text)
print(f"Length of chunks: {len(recursive_chunks)}")

print(f"First chunk: {recursive_chunks[0][:100]}")

Length of chunks: 2
First chunk: Python is a high-level programming language


Python is widely used for:
- Data Science
- Machine Le


In [44]:
print(recursive_chunks[0])
print("----------------")
print(recursive_chunks[1])
print("----------------")

Python is a high-level programming language


Python is widely used for:
- Data Science
- Machine Learning
- AI
- RAG applications

Variables:
name = "Sumona"
age = 25
----------------
Lists:
skills = ["Python", "SQL", "RAG"]
----------------


In [45]:
## Method 3 - Token Based Text Splitter

from langchain_text_splitters import TokenTextSplitter
# TokenTextSplitter ko import kar rahe hain

token_splitter = TokenTextSplitter(
    chunk_size=200,
    # Har chunk mein maximum 200 tokens honge

    chunk_overlap=20,
    # Consecutive chunks ke beech 20 tokens overlap rahenge

    length_function=len
    # Length calculate karne ke liye len() function use hoga
)

token_chunks = token_splitter.split_text(text)
# Text ko token-based chunks mein split karega

print(f"Length of chunks: {len(token_chunks)}")
# Total number of chunks

print(f"First chunk: {token_chunks[0][:100]}")
# First chunk ke first 100 characters

Length of chunks: 1
First chunk: Python is a high-level programming language


Python is widely used for:
- Data Science
- Machine Le
